In [ ]:
# ============================================================
# Week 6 BBO Capstone Script
# GP + SVM + MLP + CNN-inspired 1D Neural Network
# ============================================================

import numpy as np
import pandas as pd
import warnings

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

import torch
import torch.nn as nn
import torch.optim as optim

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)

# ============================================================
# Historical query inputs: Weeks 1–5
# Each row contains points for Functions 1–8
# ============================================================

historical_inputs = [
    # Week 1 / previous returned row
    [
        [0.761024, 0.713000],
        [0.732637, 0.906564],
        [0.522581, 0.591593, 0.350176],
        [0.564020, 0.473834, 0.390972, 0.258427],
        [0.196777, 0.892275, 0.855813, 0.891829],
        [0.728785, 0.146949, 0.767044, 0.739699, 0.020789],
        [0.016700, 0.532618, 0.280647, 0.222769, 0.407487, 0.748018],
        [0.035844, 0.064098, 0.010711, 0.024463, 0.361511, 0.783344, 0.515157, 0.880442],
    ],
    # Week 2
    [
        [0.761024, 0.713000],
        [0.732637, 0.906564],
        [0.522581, 0.591593, 0.350176],
        [0.564020, 0.473834, 0.390972, 0.258427],
        [0.196777, 0.892275, 0.855813, 0.891829],
        [0.728785, 0.146949, 0.767044, 0.739699, 0.020789],
        [0.016700, 0.532618, 0.280647, 0.222769, 0.407487, 0.748018],
        [0.035844, 0.064098, 0.010711, 0.024463, 0.361511, 0.783344, 0.515157, 0.880442],
    ],
    # Week 3
    [
        [0.751825, 0.811946],
        [0.980817, 0.626715],
        [0.996446, 0.737055, 0.850924],
        [0.404477, 0.413254, 0.303108, 0.434359],
        [0.521278, 0.505138, 0.985473, 0.994545],
        [0.341155, 0.021278, 0.626487, 0.970661, 0.032762],
        [0.431390, 0.173879, 0.071339, 0.124473, 0.403592, 0.624180],
        [0.064214, 0.412793, 0.081589, 0.008195, 0.974438, 0.216196, 0.139173, 0.110624],
    ],
    # Week 4
    [
        [0.702630, 0.955980],
        [0.700329, 1.000000],
        [0.977206, 0.330727, 0.000017],
        [0.393932, 0.393230, 0.377559, 0.426151],
        [0.449325, 0.603715, 1.000000, 1.000000],
        [0.941155, 0.654101, 0.079248, 0.961137, 0.229746],
        [0.153100, 0.236737, 0.190018, 0.000000, 0.365128, 0.744155],
        [0.103762, 0.017606, 0.204751, 0.194101, 0.745367, 0.674182, 0.125362, 0.044024],
    ],
    # Week 5 latest submitted inputs
    [
        [0.012715, 0.992415],
        [0.026867, 0.999598],
        [0.955194, 0.008179, 0.700062],
        [0.425212, 0.377719, 0.427527, 0.420346],
        [0.479838, 0.890814, 1.000000, 1.000000],
        [0.137088, 0.268975, 0.283122, 0.986752, 0.045377],
        [0.000000, 0.450181, 0.211028, 0.000000, 0.353842, 1.000000],
        [0.000000, 0.000000, 0.301242, 0.216500, 0.000000, 1.000000, 0.000000, 1.000000],
    ],
]

historical_outputs = [
    [-7.565331332744927e-18, 0.38412771439907634, -0.04757757182509677, -4.83054096204485, 1257.680268889983, -0.7671825918815833, 1.2394822144938658, 9.5430069620611],
    [-7.565331332744927e-18, 0.5530064492925906, -0.03333218343962805, -4.83054096204485, 1257.680268889983, -0.8072367077314392, 1.2394822144938658, 9.5430069620611],
    [2.495129202697582e-35, 0.06646691679004207, -0.04126707799435369, -0.7158150747637886, 1769.2992166742467, -0.721361811601727, 1.493591747104962, 9.7741312776374],
    [-8.189634555426656e-79, 0.5980717829505922, -0.12802055717541724, 0.2735224699683667, 2027.715300336871, -1.6878746934171067, 1.120167075371798, 9.8898511444579],
    [0.0, 0.008971860858651539, -0.18597539887299502, 0.5994256847352308, 3420.1124772143226, -1.0804901380041434, 0.3185961294770624, 9.199106282308],
]

# ============================================================
# CNN Surrogate Model
# ============================================================

class CNN1DSurrogate(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(8, 16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(16 * input_dim, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        return self.fc2(x).squeeze(-1)

def train_cnn(X, y, epochs=500, lr=0.01):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_mean = np.mean(y)
    y_std = np.std(y) if np.std(y) > 1e-12 else 1.0
    y_scaled = (y - y_mean) / y_std
    y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

    model = CNN1DSurrogate(X.shape[1])
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        pred = model(X_tensor)
        loss = loss_fn(pred, y_tensor)
        loss.backward()
        optimizer.step()

    return model, y_mean, y_std

def cnn_predict(model, X, y_mean, y_std):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X, dtype=torch.float32)
        pred_scaled = model(X_tensor).numpy()
    return pred_scaled * y_std + y_mean

def cnn_gradient(model, x):
    model.eval()
    x_tensor = torch.tensor(x.reshape(1, -1), dtype=torch.float32, requires_grad=True)
    pred = model(x_tensor)
    pred.backward()
    grad = x_tensor.grad.detach().numpy().ravel()
    return grad

def cnn_feature_importance(model, X):
    grads = []
    for x in X:
        g = np.abs(cnn_gradient(model, x))
        grads.append(g)

    importance = np.mean(grads, axis=0)
    if importance.sum() > 0:
        importance = importance / importance.sum()
    return importance

def normalize_score(values):
    values = np.array(values)
    std = np.std(values)
    if std < 1e-12:
        return np.zeros_like(values)
    return (values - np.mean(values)) / std

def pair_exists(X, y, point, output):
    same_x = np.all(np.isclose(X, point.reshape(1, -1), atol=1e-10), axis=1)
    same_y = np.isclose(y, output, atol=1e-10)
    return np.any(same_x & same_y)

# ============================================================
# Main Loop
# ============================================================

results = []
importance_rows = []

for func_id in range(1, 9):

    print("\n" + "=" * 60)
    print(f"Function {func_id} - Week 6")
    print("=" * 60)

    X = np.load(f"function{func_id}_inputs.npy")
    y = np.load(f"function{func_id}_outputs.npy")

    # Append all historical query points/outputs
    for r in range(len(historical_inputs)):
        point = np.array(historical_inputs[r][func_id - 1], dtype=float)
        output = float(historical_outputs[r][func_id - 1])

        if point.shape[0] != X.shape[1]:
            raise ValueError(f"Function {func_id}: dimension mismatch.")

        if not pair_exists(X, y, point, output):
            X = np.vstack([X, point])
            y = np.append(y, output)

    dim = X.shape[1]

    best_idx = np.argmax(y)
    best_input = X[best_idx]
    best_output = y[best_idx]

    print("Dataset size:", X.shape)
    print("Best output:", best_output)
    print("Best input:", best_input)

    # -----------------------------
    # GP model
    # -----------------------------
    kernel = ConstantKernel(1.0) * RBF(length_scale=np.ones(dim)) + WhiteKernel(noise_level=1e-5)
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, alpha=1e-6, random_state=42)
    gp.fit(X, y)

    # -----------------------------
    # SVM promising-region classifier
    # -----------------------------
    threshold = np.quantile(y, 0.70)
    labels = (y >= threshold).astype(int)

    if len(np.unique(labels)) > 1:
        svm = Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(kernel="rbf", probability=True, C=1.0, gamma="scale", random_state=42))
        ])
        svm.fit(X, labels)
        use_svm = True
    else:
        svm = None
        use_svm = False

    # -----------------------------
    # MLP surrogate
    # -----------------------------
    mlp = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(128, 64, 32),
            activation="relu",
            alpha=0.0005,
            learning_rate_init=0.001,
            max_iter=1200,
            random_state=42
        ))
    ])
    mlp.fit(X, y)

    # -----------------------------
    # CNN-inspired surrogate
    # -----------------------------
    cnn_model, y_mean, y_std = train_cnn(X, y, epochs=600, lr=0.01)
    cnn_importance = cnn_feature_importance(cnn_model, X)

    print("CNN feature importance:", cnn_importance)

    imp_row = {"Function": func_id}
    for j in range(dim):
        imp_row[f"x{j+1}_importance"] = cnn_importance[j]
    importance_rows.append(imp_row)

    # -----------------------------
    # Candidate generation
    # -----------------------------
    if dim <= 3:
        n_global = 7000
        n_local = 2500
        beta = 1.5
        noise = 0.03
        grad_lr = 0.035
        grad_steps = 25
    elif dim <= 5:
        n_global = 10000
        n_local = 3000
        beta = 2.0
        noise = 0.05
        grad_lr = 0.045
        grad_steps = 30
    else:
        n_global = 15000
        n_local = 3500
        beta = 2.5
        noise = 0.08
        grad_lr = 0.055
        grad_steps = 35

    global_candidates = np.random.uniform(0, 1, size=(n_global, dim))
    local_candidates = np.clip(best_input + np.random.normal(0, noise, size=(n_local, dim)), 0, 1)

    # CNN gradient ascent from top 5 observed points
    top_k = min(5, len(y))
    top_indices = np.argsort(y)[-top_k:]
    grad_candidates = []

    for idx in top_indices:
        x = X[idx].copy()

        for _ in range(grad_steps):
            grad = cnn_gradient(cnn_model, x)
            grad = grad * (0.5 + cnn_importance)

            norm = np.linalg.norm(grad)
            if norm > 1e-12:
                grad = grad / norm

            x = np.clip(x + grad_lr * grad, 0, 1)

        grad_candidates.append(x)

    grad_candidates = np.array(grad_candidates)

    candidates = np.vstack([global_candidates, local_candidates, grad_candidates])

    # -----------------------------
    # Scoring
    # -----------------------------
    mu, sigma = gp.predict(candidates, return_std=True)
    ucb = mu + beta * sigma

    mlp_pred = mlp.predict(candidates)
    cnn_pred = cnn_predict(cnn_model, candidates, y_mean, y_std)

    if use_svm:
        svm_prob = svm.predict_proba(candidates)[:, 1]
    else:
        svm_prob = np.ones(len(candidates))

    grad_strength = []
    for c in candidates:
        g = cnn_gradient(cnn_model, c)
        grad_strength.append(np.linalg.norm(g))
    grad_strength = np.array(grad_strength)

    final_score = (
        0.30 * normalize_score(ucb)
        + 0.25 * normalize_score(mlp_pred)
        + 0.25 * normalize_score(cnn_pred)
        + 0.10 * svm_prob
        + 0.10 * normalize_score(grad_strength)
    )

    best_candidate_idx = np.argmax(final_score)
    new_point = np.clip(candidates[best_candidate_idx], 0, 1)

    query = "-".join([f"{v:.6f}" for v in new_point])

    print("Week 6 Submission:", query)

    results.append({
        "Function": func_id,
        "Dimension": dim,
        "Best_Output": float(best_output),
        "Week6_Query": query
    })

# ============================================================
# Save Results
# ============================================================

pd.DataFrame(results).to_csv("Week6_Submissions.csv", index=False)
pd.DataFrame(importance_rows).to_csv("Week6_CNN_Feature_Importance.csv", index=False)

print("\nDone")
print("Saved: Week6_Submissions.csv")
print("Saved: Week6_CNN_Feature_Importance.csv")


Function 1 - Week 6
Dataset size: (14, 2)
Best output: 7.710875114502849e-16
Best input: [0.73102363 0.73299988]
CNN feature importance: [0.47118253 0.5288175 ]
Week 6 Submission: 0.012658-0.001204

Function 2 - Week 6
Dataset size: (15, 2)
Best output: 0.6112052157614438
Best input: [0.70263656 0.9265642 ]
CNN feature importance: [0.51628155 0.48371843]
Week 6 Submission: 0.803770-0.997320

Function 3 - Week 6
Dataset size: (20, 3)
Best output: -0.03333218343962805
Best input: [0.522581 0.591593 0.350176]
CNN feature importance: [0.31181437 0.34544435 0.34274125]
Week 6 Submission: 0.034249-0.018981-0.451567

Function 4 - Week 6
Dataset size: (34, 4)
Best output: 0.5994256847352308
Best input: [0.425212 0.377719 0.427527 0.420346]
CNN feature importance: [0.25671038 0.31612837 0.19684221 0.23031905]
Week 6 Submission: 0.485961-0.345148-0.465576-0.429334

Function 5 - Week 6
Dataset size: (24, 4)
Best output: 3420.1124772143226
Best input: [0.479838 0.890814 1.       1.      ]
CNN fea